# ROI por producto: BD vs modelo

Para cada **producto** comparamos el retorno de la inversión (ROI):
- **Beneficio** = (nº de clicks) × **CPL**. El CPL es el ingreso que deja cada click y es propio de cada producto.
- **Coste**: enviar emails tiene un coste. Lo analizamos con **tres estructuras de coste**, todas **por email enviado**:
  - **Fijo:** cada email cuesta lo mismo (1 €), igual para todos los productos.
  - **Variable:** cada email cuesta un **% del CPL** del producto (20% / 30% / 40%).
  - **Mixto:** una base fija por email **más** un % del CPL (0,50 € + 15% del CPL).
- **ROI** = (beneficio − coste) / coste.

Comparamos dos formas de hacer la campaña:
- **BD (lo que había):** se envió el producto a todos los usuarios registrados.
- **Nuestro modelo:** envía un email solo si su beneficio esperado (`p_model × CPL`) supera su coste marginal, priorizando a quién es probable que clique.

> CPL faltante (registros sin CPL informado) imputado con la mediana (€8).

## 1 · Datos: envíos, clicks y CPL por evento

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
sns.set_style('whitegrid')
def find_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p/'pyproject.toml').exists() or (p/'.git').exists():
            return p
    return Path.cwd()
PROC = find_root()/'data'/'processed'

prop = pd.read_csv(PROC/'propensity_scores.csv')
prods = pd.read_csv(PROC/'products.csv')[['id_product','cpl']]
# CPL por evento (beneficio por click); imputamos los faltantes con la mediana
prop['cpl'] = pd.to_numeric(prop['id_product'].map(prods.set_index('id_product')['cpl']), errors='coerce')
prop['cpl'] = prop['cpl'].fillna(prop['cpl'].median())
prop['beneficio'] = prop['target'] * prop['cpl']            # beneficio real (solo si hubo click)
prop['beneficio_esperado'] = prop['p_model'] * prop['cpl']  # beneficio esperado (para decidir a quién enviar)
print('Eventos:', len(prop), '| clicks:', int(prop.target.sum()), '| CPL medio:', round(prop.cpl.mean(), 2))

## 2 · Parámetros de coste (los tres escenarios)

Todos los costes son **por email enviado** (coste marginal):
- **Fijo:** `C_FIJO_EMAIL` € por email, igual para todos los productos.
- **Variable:** un porcentaje del CPL del producto (`PCT_VAR`). Comparamos 20% / 30% / 40%.
- **Mixto:** base fija `BASE_MIX` € más un porcentaje `PCT_MIX` del CPL por email.

> Regla de decisión del modelo: enviar si `p_model × CPL ≥ coste_marginal`. En el caso **variable** esto equivale exactamente a `p_model ≥ porcentaje`.

In [ ]:
C_FIJO_EMAIL = 1.0              # FIJO: coste por email, igual para todos los productos (EUR)
PCT_VAR = [0.20, 0.30, 0.40]   # VARIABLE: coste por email = % del CPL (tres escenarios a comparar)
BASE_MIX, PCT_MIX = 0.50, 0.15 # MIXTO: base fija (EUR) + % del CPL por email

def coste_marginal(escenario, cpl):
    """Coste de enviar UN email (por fila), según la estructura de coste y el CPL del producto."""
    if escenario == 'fijo':
        return pd.Series(C_FIJO_EMAIL, index=cpl.index)
    if escenario.startswith('variable_'):
        pct = float(escenario.split('_')[1]) / 100
        return pct * cpl
    if escenario == 'mixto':
        return BASE_MIX + PCT_MIX * cpl
    raise ValueError(escenario)

ESCENARIOS = ['fijo'] + [f'variable_{int(p*100)}' for p in PCT_VAR] + ['mixto']
print('Escenarios de coste definidos:', ESCENARIOS)

## 3 · ROI por producto: BD vs modelo, en cada escenario de coste

In [ ]:
def roi_por_producto(escenario):
    filas = []
    for producto, sub in prop.groupby('product_new'):
        cm = coste_marginal(escenario, sub['cpl'])   # coste por email (vector, una fila por envío)
        n = len(sub)
        clicks = int(sub['target'].sum())
        beneficio = sub['beneficio'].sum()
        # --- BD: se envió a todos ---
        coste_bd = cm.sum()
        roi_bd = (beneficio - coste_bd) / coste_bd if coste_bd > 0 else np.nan
        # --- Modelo: enviar un email solo si su beneficio esperado >= su coste marginal ---
        mask = sub['beneficio_esperado'] >= cm
        n_mod = int(mask.sum())
        beneficio_mod = sub.loc[mask, 'beneficio'].sum()
        coste_mod = cm[mask].sum()
        roi_mod = (beneficio_mod - coste_mod) / coste_mod if coste_mod > 0 else np.nan
        filas.append({'producto': producto, 'cpl': round(sub['cpl'].mean(), 1),
                      'envios': n, 'clicks': clicks,
                      'ROI_BD_%': round(100 * roi_bd, 0),
                      'envios_modelo': n_mod,
                      'ROI_modelo_%': round(100 * roi_mod, 0) if not np.isnan(roi_mod) else np.nan})
    df = pd.DataFrame(filas).sort_values('envios', ascending=False).reset_index(drop=True)
    df['mejora'] = df['ROI_modelo_%'] - df['ROI_BD_%']
    return df

tablas = {}
for escenario in ESCENARIOS:
    d = roi_por_producto(escenario)
    tablas[escenario] = d
    n_mejora = int((d['mejora'] > 0).sum())
    print(f'--- COSTE {escenario.upper()}: el modelo mejora el ROI en {n_mejora}/{len(d)} productos ---')
    display(d)

In [ ]:
# --- ROI AGREGADO por escenario (suma de beneficios y costes sobre TODOS los productos) ---
def roi_agregado(escenario):
    cm = coste_marginal(escenario, prop['cpl'])            # coste por email (una fila por envío)
    beneficio_total = prop['beneficio'].sum()
    coste_bd = cm.sum()                                     # BD: se envía a todos
    roi_bd = (beneficio_total - coste_bd) / coste_bd
    mask = prop['beneficio_esperado'] >= cm                # modelo: enviar si beneficio esperado >= coste
    beneficio_mod = prop.loc[mask, 'beneficio'].sum()
    coste_mod = cm[mask].sum()
    roi_mod = (beneficio_mod - coste_mod) / coste_mod if coste_mod > 0 else float('nan')
    n_mejora = int((tablas[escenario]['mejora'] > 0).sum())
    return {'escenario': escenario,
            'ROI_BD_%': round(100*roi_bd, 0),
            'ROI_modelo_%': round(100*roi_mod, 0),
            'emails_enviados_%': round(100*mask.sum()/len(prop), 1),
            'productos_mejoran': f'{n_mejora}/{len(tablas[escenario])}'}

roi_agg = pd.DataFrame([roi_agregado(e) for e in ESCENARIOS])
print('=== ROI AGREGADO por estructura de coste (modelo plano) ===')
print(roi_agg.to_string(index=False))


## 3.bis · ROI a ESCALA DE PRODUCCIÓN (prior real 2%)

El ROI anterior se mide sobre la muestra **balanceada** (24% de clic). Para estimar el ROI **real** de
producción reponderamos cada fila al prior real (2%) y decidimos con la probabilidad **corregida** `p_real`
(beneficio esperado realista = `p_real × CPL`). Es la lectura económica a escala real.

In [ ]:
# Reponderacion al prior real: pesos que llevan la prevalencia de 0.238 (balanceada) a 0.02 (produccion).
rho_train = float(prop['target'].mean()); rho_real = 0.02
prop['w'] = np.where(prop['target']==1, rho_real/rho_train, (1-rho_real)/(1-rho_train))
print(f"prevalencia reponderada = {(prop['w']*prop['target']).sum()/prop['w'].sum():.4f}  |",
      f"p_real: media={prop['p_real'].mean():.4f} p90={prop['p_real'].quantile(0.9):.4f} max={prop['p_real'].max():.4f}")

def roi_produccion(escenario):
    cm = coste_marginal(escenario, prop['cpl'])
    ben = (prop['w']*prop['target']*prop['cpl']).sum()              # beneficio reponderado (solo clics)
    coste_bd = (prop['w']*cm).sum()
    roi_bd = (ben-coste_bd)/coste_bd
    mask = prop['p_real']*prop['cpl'] >= cm                          # decision con prob REALISTA
    ben_m = (prop.loc[mask,'w']*prop.loc[mask,'target']*prop.loc[mask,'cpl']).sum()
    coste_m = (prop.loc[mask,'w']*cm[mask]).sum()
    roi_m = (ben_m-coste_m)/coste_m if coste_m>0 else float('nan')
    pct = 100*prop.loc[mask,'w'].sum()/prop['w'].sum()
    return {'escenario': escenario, 'ROI_BD_%': round(100*roi_bd),
            'ROI_modelo_%': round(100*roi_m) if not np.isnan(roi_m) else None,
            'emails_%': round(pct, 1)}

prod = pd.DataFrame([roi_produccion(e) for e in ESCENARIOS])
print('=== ROI a escala de PRODUCCION (prior real 2%) ===')
print(prod.to_string(index=False))
print('-> A 2% real, enviar a todos pierde dinero siempre; ni el targeting rescata el canal:')
print('   el mejor caso (coste fijo bajo) queda en break-even (~0%); domina la economia unitaria.')

## 4 · Visualización (coste variable 30%): ROI por producto, BD vs modelo

In [ ]:
d = tablas['variable_30'].sort_values('ROI_BD_%')
y = np.arange(len(d)); h = 0.4
fig, ax = plt.subplots(figsize=(9, 8))
ax.barh(y + h/2, d['ROI_BD_%'], h, label='BD (enviar a todos)', color='#bdbdbd')
ax.barh(y - h/2, d['ROI_modelo_%'], h, label='Modelo (priorizar)', color='#d1495b')
ax.axvline(0, color='gray', lw=0.8)
ax.set_yticks(y); ax.set_yticklabels(d['producto'], fontsize=8)
ax.set_xlabel('ROI (%)'); ax.set_title('ROI por producto (coste variable 30%): BD vs modelo')
ax.legend()
plt.tight_layout(); plt.show()

## 5 · Conclusiones

Con todas las estructuras de coste **por email**, priorizar con el modelo **mejora el ROI global** frente a enviar a todos (BD). La regla de envío es `p_model × CPL ≥ coste_marginal`.

**Resumen global (ROI BD → ROI modelo):**

| Estructura de coste | ROI BD | ROI modelo | Emails enviados | Clicks retenidos | Productos que mejoran |
|---|---|---|---|---|---|
| Fijo (1 €/email) | 101% | **322%** | 13,2% | 23,5% | 19/25 |
| Variable 20% CPL | 18% | **143%** | 2,9% | 5,9% | 21/25 |
| Variable 30% CPL | −21% | **87%** | 0,5% | 1,1% | 16/25 |
| Variable 40% CPL | −41% | **102%** | 0,1% | 0,4% | 11/25 |
| Mixto (0,50 € + 15% CPL) | 13% | **132%** | 2,3% | 4,8% | 21/25 |

**Lecturas:**
- **Fijo (1 €):** el modelo triplica el ROI (101% → 322%) enviando solo al 13% de mayor propensión. Es el escenario con mejor equilibrio ROI/volumen (conserva ~24% de los clicks).
- **Variable (% del CPL):** la regla se reduce a `p_model ≥ %`. Cuanto mayor el porcentaje, más exigente el umbral y menos se envía. Enviar a todos (BD) ya es deficitario a 30–40% (ROI negativo), porque el coste por email es alto frente a la tasa real de click (~24%); el modelo lo rescata a positivo, pero a 30–40% envía tan poco (<1%) que el negocio es testimonial. A 20% está el punto razonable (ROI 143% y aún captura ~6% de los clicks).
- **Mixto (0,50 € + 15% CPL):** intermedio; el modelo multiplica el ROI por ~10 (13% → 132%) enviando al 2,3%.

**Trade-off:** el modelo sube el ROI reduciendo drásticamente el volumen (descarta los clicks de baja propensión). El ROI muy alto en variable 30–40% se logra sobre poquísimos envíos: si el objetivo es maximizar clicks totales y no la eficiencia por euro, conviene un porcentaje bajo (o coste fijo) con umbral menos exigente. La elección depende del objetivo de negocio: eficiencia (€ por click) frente a alcance (clicks totales).

> Nota: CPL imputado a la mediana (€8) en los registros sin CPL informado; el coste fijo, los porcentajes variables y la combinación mixta son parámetros editables en la sección 2.

## 6 · Nota sobre el prior real

Se distinguen **dos lecturas** del ROI de este notebook:

- **ROI sobre la muestra balanceada** (secciones 3-5): compara *en términos relativos* "enviar a
  todos" frente a "priorizar con el modelo", usando `p_model` (escala de entrenamiento). El **orden**
  de los productos por ROI no depende del prior real, porque la corrección de prior es monótona
  (cambia el nivel de las probabilidades, no su orden).
- **ROI a escala de producción** (sección 3.bis): repondera la muestra al prior real (~2 %) y decide
  con la probabilidad corregida `p_real`. Aquí el **nivel** del ROI en euros sí depende del prior, y a
  una tasa real del 2 % el panorama es mucho más exigente: enviar a todos pierde dinero en todas las
  estructuras y el modelo solo rescata a positivo el coste fijo bajo (1 € → +14 %).

En resumen: lo que es **invariante** al prior es el *ranking* de productos por ROI; el *nivel* absoluto
del ROI sí cambia con el prior, y por eso se analiza explícitamente en la sección 3.bis.